# 02 — Nettoyage, imputation contrôlée et variables retardées

Ce notebook prépare une table régulière de 15 minutes à partir des exports SINERT. Les fichiers de `data/raw/` ne sont jamais modifiés. Les petits trous sont interpolés uniquement lorsqu'ils sont entourés de mesures ; les longues périodes manquantes restent absentes.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_sinert_measurements
from src.features import add_calendar_features, add_lag_features
from src.preprocessing import (
    add_estimated_power_factor,
    combine_date_and_time,
    impute_short_gaps,
    regularize_time_grid,
    remove_duplicates,
)

MEASUREMENT_COLUMNS = [
    "EM_VIRTUAL kWh (kWh) [ABS]",
    "EM_VIRTUAL kvarh (kvarh) [ABS]",
    "EM_VIRTUAL kW dmd (kW) [MAX]",
    "EM_VIRTUAL kVA dmd (kVA) [MAX]",
    "EM_VIRTUAL kW sys (kW) [AVG]",
    "EM_VIRTUAL kVA sys (kVA) [AVG]",
    "power_factor_estimated",
]
LAGS = (1, 4, 96, 672)


In [2]:
raw = load_sinert_measurements(
    PROJECT_ROOT / "data" / "raw" / "SINERT_DATA_CONCENTRATOR"
)
frame = combine_date_and_time(raw)
invalid_timestamps = int(frame["timestamp"].isna().sum())
frame = add_estimated_power_factor(frame)
frame = remove_duplicates(frame, "timestamp")
frame = frame.sort_values("timestamp").reset_index(drop=True)
print(f"Lignes chargées : {len(raw):,}")
print(f"Horodatages invalides : {invalid_timestamps:,}")
print(f"Doublons supprimés : {len(raw) - invalid_timestamps - len(frame):,}")
print(f"Lignes temporelles uniques : {len(frame):,}")
print(f"Période : {frame['timestamp'].min()} → {frame['timestamp'].max()}")


Lignes chargées : 152,872
Horodatages invalides : 0
Doublons supprimés : 3,878
Lignes temporelles uniques : 148,994
Période : 2022-06-01 00:00:00 → 2026-09-01 00:00:00


In [3]:
def missing_run_summary(dataframe, columns):
    summaries = []
    for column in columns:
        missing = dataframe[column].isna()
        groups = missing.ne(missing.shift()).cumsum()
        lengths = missing.groupby(groups).sum()
        lengths = lengths[lengths > 0]
        summaries.append({
            "column": column,
            "missing_before": int(missing.sum()),
            "max_consecutive_missing": int(lengths.max()) if not lengths.empty else 0,
            "runs_longer_than_4": int((lengths > 4).sum()),
        })
    return pd.DataFrame(summaries)

missing_before = missing_run_summary(frame, MEASUREMENT_COLUMNS)
missing_before


,column,missing_before,max_consecutive_missing,runs_longer_than_4
0,EM_VIRTUAL kWh (kWh) [ABS],3704,2592,25
1,EM_VIRTUAL kvarh (kvarh) [ABS],3704,2592,25
2,EM_VIRTUAL kW dmd (kW) [MAX],3703,2592,24
3,EM_VIRTUAL kVA dmd (kVA) [MAX],3703,2592,24
4,EM_VIRTUAL kW sys (kW) [AVG],3703,2592,24
5,EM_VIRTUAL kVA sys (kVA) [AVG],3703,2592,24
6,power_factor_estimated,3703,2592,24


In [4]:
# Une grille régulière évite qu'un lag traverse silencieusement une journée absente.
prepared = frame[["timestamp", *MEASUREMENT_COLUMNS, "source_file"]]
prepared = regularize_time_grid(prepared, "timestamp", frequency="15min")
prepared = impute_short_gaps(
    prepared,
    "timestamp",
    columns=MEASUREMENT_COLUMNS,
    max_consecutive_missing=4,
)

missing_after = prepared[MEASUREMENT_COLUMNS].isna().sum()
imputed_counts = (
    frame.set_index("timestamp")[MEASUREMENT_COLUMNS]
    .reindex(prepared.set_index("timestamp").index)
    .isna()
)
print("Valeurs manquantes après interpolation des trous de 1 heure maximum :")
print(missing_after.to_string())
print(f"Nombre de points sur la grille régulière : {len(prepared):,}")


Valeurs manquantes après interpolation des trous de 1 heure maximum :
EM_VIRTUAL kWh (kWh) [ABS]        3013
EM_VIRTUAL kvarh (kvarh) [ABS]    3013
EM_VIRTUAL kW dmd (kW) [MAX]      3006
EM_VIRTUAL kVA dmd (kVA) [MAX]    3006
EM_VIRTUAL kW sys (kW) [AVG]      3006
EM_VIRTUAL kVA sys (kVA) [AVG]    3006
power_factor_estimated            3006
Nombre de points sur la grille régulière : 149,089


In [5]:
prepared = add_calendar_features(prepared, "timestamp")
prepared = add_lag_features(prepared, columns=MEASUREMENT_COLUMNS, lags=LAGS)
lag_columns = [f"{column}_lag_{lag}" for column in MEASUREMENT_COLUMNS for lag in LAGS]
model_columns = [*MEASUREMENT_COLUMNS, *lag_columns]
model_frame = prepared.dropna(subset=model_columns).copy()
model_frame = model_frame.sort_values("timestamp").reset_index(drop=True)
print(f"Variables retardées créées : {len(lag_columns)}")
print(f"Lignes avant suppression des trous longs : {len(prepared):,}")
print(f"Lignes utilisables pour la modélisation : {len(model_frame):,}")
print(f"Lignes écartées : {len(prepared) - len(model_frame):,}")
print("Exemple de colonnes :", [column for column in model_frame.columns if "lag_" in column][:8])


Variables retardées créées : 28
Lignes avant suppression des trous longs : 149,089
Lignes utilisables pour la modélisation : 144,513
Lignes écartées : 4,576
Exemple de colonnes : ['EM_VIRTUAL kWh (kWh) [ABS]_lag_1', 'EM_VIRTUAL kWh (kWh) [ABS]_lag_4', 'EM_VIRTUAL kWh (kWh) [ABS]_lag_96', 'EM_VIRTUAL kWh (kWh) [ABS]_lag_672', 'EM_VIRTUAL kvarh (kvarh) [ABS]_lag_1', 'EM_VIRTUAL kvarh (kvarh) [ABS]_lag_4', 'EM_VIRTUAL kvarh (kvarh) [ABS]_lag_96', 'EM_VIRTUAL kvarh (kvarh) [ABS]_lag_672']


In [6]:
output_path = PROJECT_ROOT / "data" / "processed" / "sinert_features.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
model_frame.to_csv(output_path, index=False)
print(f"Fichier enregistré : {output_path}")
print(f"Taille : {output_path.stat().st_size / 1024 / 1024:.1f} MiB")


Fichier enregistré : c:\Évaluation approfondie2\data\processed\sinert_features.csv
Taille : 74.4 MiB
